# 12 Similarity Model Comparison

## Purpose
This notebook compares notebook-8 similarity outputs across multiple model states that were run on the same tokenizer family and experiment.

## Why this notebook matters
Notebook 8 describes one model at a time. This notebook asks the next question: when the model state changes, do the full-space distributions and the query-specific neighborhoods change in meaningful ways?

## Inputs
- Saved notebook-8 output folders for each model being compared
- An optional labeled-glycan table for neighborhood label-overlap summaries

## Outputs
- Side-by-side all-vs-all and specific-vs-all comparison tables
- Top-neighbor overlap summaries across models
- Optional label-overlap summaries for matched neighborhoods
- A self-contained HTML comparison report and an optional clean public-export folder


## Setup note

This notebook is report-oriented rather than model-training-oriented:
- code lives in GitHub
- notebook-8 outputs live in Google Drive
- the notebook reads saved CSVs and plots instead of recomputing embeddings

Because the notebook works entirely from saved notebook-8 artifacts, it can run in a CPU-only Colab session.


## Runtime setup

This cell prepares the Colab runtime by mounting Google Drive, synchronizing the GitHub repository, and making the checked-out repository importable.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the repository was cloned or updated
- the active repository directory inside the Colab runtime


In [ ]:
# Standard library imports used only for Colab runtime setup.
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', GITHUB_REF],
        check=True,
    )

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')


## Import notebook dependencies

This cell imports the shared helper modules used to assemble the comparison tables and reports.

**Expected output**
- no printed output under normal conditions
- a normal Python import error only if the repository sync step failed or a required dependency is missing


In [ ]:
import importlib
from pathlib import Path

from IPython.display import HTML, Image, display

import src.similarity_model_comparison as similarity_model_comparison
importlib.reload(similarity_model_comparison)

from src.notebook_utils import (
    SUPPORTED_TOKENIZER_FAMILIES,
    stringify_path_values,
    validate_tokenizer_family,
    write_json,
)
from src.similarity import (
    build_public_export_dir,
    build_public_report_subdir,
    export_public_similarity_model_comparison_html,
)
from src.similarity_model_comparison import build_similarity_model_comparison


## User settings

This is the main cell to review before running the notebook. Keep model-selection edits here so the comparison logic below can stay stable.

**Settings to review**
- `DRIVE_ROOT`: the root Drive folder for the project
- `TOKENIZER_FAMILY`, `EXPERIMENT_NAME`, and `OUTPUT_RUN_LABEL`: which notebook-8 run family to compare
- classifier run labels: which classification runs should be compared against the pretrained baseline
- `LABEL_TABLE_PATH`: whether semantic label-overlap summaries should be computed
- `COMPARISON_RUN_LABEL`, neighborhood settings, and export settings: how outputs are named and summarized

**Expected output**
- this cell only defines settings; it does not run the comparison


In [ ]:
from pathlib import Path

# Update DRIVE_ROOT if the project folder uses a different Google Drive path.
DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Choose the notebook-8 run family that should be compared across models.
TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
OUTPUT_RUN_LABEL = 'live_extended'

# Set the classification run labels that correspond to the two classifier states.
CLASSIFIER_MLM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_mlm'
CLASSIFIER_MLM_EVAL_LABEL = 'cls_lr2e-5_ep100_bs16_mlm'
CLASSIFIER_RANDOM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_randominit'
CLASSIFIER_RANDOM_EVAL_LABEL = 'cls_lr2e-5_ep100_bs16_randominit_eval'

# Provide the prepared classification table if neighborhood label-overlap summaries should be created.
LABEL_TABLE_PATH = DRIVE_ROOT / 'results' / 'classification_prep' / 'prepared_classification_rows.csv'

# Name this comparison run and configure the comparison summaries.
COMPARISON_RUN_LABEL = 'pretrain_vs_classifier_mlm_vs_randominit'
CLOUD_THRESHOLDS = [0.90, 0.85, 0.80]
TOP_K_NEIGHBORS = 25
HTML_CLOUD_THRESHOLD = 0.90
HTML_TOP_N_NEIGHBORS = 8
HTML_REPORT_TITLE = f'{TOKENIZER_FAMILY} similarity model comparison'
EMBED_HTML_IMAGES = True

# Configure the optional clean public-export folder.
PUBLIC_EXPORT_ENABLED = True
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True


## Resolve run directories and save the notebook configuration

This cell turns the editable settings into concrete input and output paths, builds the run-spec table that drives the comparison, and saves a small JSON configuration record next to the outputs.

**Expected output**
- the resolved comparison output directory
- one line for each run directory being compared
- the public-export destination for a later manual copy step

**How to interpret the result**
- if a run directory does not exist, review the tokenizer, experiment, output-run, or classifier settings before continuing


In [ ]:
validate_tokenizer_family(TOKENIZER_FAMILY, supported_families=SUPPORTED_TOKENIZER_FAMILIES)

SIMILARITY_SCALEUP_ROOT = DRIVE_ROOT / 'results' / 'similarity_scaleup'
CLASSIFICATION_SCALEUP_ROOT = SIMILARITY_SCALEUP_ROOT / 'classification'
CLASSIFICATION_EVALUATION_ROOT = DRIVE_ROOT / 'results' / 'classification_evaluation'

# Build the exact notebook-8 run directories that will be compared.
RUN_SPECS = [
    {
        'model_id': 'pretrained_mlm',
        'model_label': 'Pretrained MLM',
        'run_dir': SIMILARITY_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / OUTPUT_RUN_LABEL,
    },
    {
        'model_id': 'classification_mlm_init',
        'model_label': 'Classifier, MLM init',
        'run_dir': CLASSIFICATION_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / CLASSIFIER_MLM_RUN_LABEL / OUTPUT_RUN_LABEL,
        'classification_evaluation_dir': CLASSIFICATION_EVALUATION_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / CLASSIFIER_MLM_EVAL_LABEL,
    },
    {
        'model_id': 'classification_random_init',
        'model_label': 'Classifier, random init',
        'run_dir': CLASSIFICATION_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / CLASSIFIER_RANDOM_RUN_LABEL / OUTPUT_RUN_LABEL,
        'classification_evaluation_dir': CLASSIFICATION_EVALUATION_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / CLASSIFIER_RANDOM_EVAL_LABEL,
    },
]

OUTPUT_DIR = (
    DRIVE_ROOT
    / 'results'
    / 'similarity_model_comparison'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
    / COMPARISON_RUN_LABEL
)
PUBLIC_REPORT_NOTEBOOK_STEM = '12_glyberta_similarity_model_comparison'
PUBLIC_EXPORT_PATH_PARTS = [TOKENIZER_FAMILY, EXPERIMENT_NAME, COMPARISON_RUN_LABEL]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)

comparison_config = {
    'drive_root': str(DRIVE_ROOT),
    'tokenizer_family': TOKENIZER_FAMILY,
    'experiment_name': EXPERIMENT_NAME,
    'output_run_label': OUTPUT_RUN_LABEL,
    'label_table_path': str(LABEL_TABLE_PATH),
    'output_dir': str(OUTPUT_DIR),
    'cloud_thresholds': CLOUD_THRESHOLDS,
    'top_k_neighbors': TOP_K_NEIGHBORS,
    'html_cloud_threshold': HTML_CLOUD_THRESHOLD,
    'html_top_n_neighbors': HTML_TOP_N_NEIGHBORS,
    'html_report_title': HTML_REPORT_TITLE,
    'embed_html_images': EMBED_HTML_IMAGES,
    'run_specs': [stringify_path_values(spec) for spec in RUN_SPECS],
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_json(OUTPUT_DIR / 'similarity_model_comparison_config.json', comparison_config)

print(f'Comparison output dir: {OUTPUT_DIR}')
for spec in RUN_SPECS:
    run_dir = Path(spec['run_dir'])
    print(f'- {spec["model_label"]}: {run_dir} | exists={run_dir.exists()}')
    print(f'  all_vs_all_summary.csv exists={(run_dir / "all_vs_all_summary.csv").exists()}')
    if 'classification_evaluation_dir' in spec:
        print(
            f'  evaluation: {spec["classification_evaluation_dir"]} | '
            f'exists={Path(spec["classification_evaluation_dir"]).exists()}'
        )

print(f'Label table: {LABEL_TABLE_PATH} | exists={LABEL_TABLE_PATH.exists()}')
print(f'Public export enabled: {PUBLIC_EXPORT_ENABLED}')
print(f'Public export Drive folder: {PUBLIC_EXPORT_DIR}')
print(f'Repo destination after copy: {PUBLIC_EXPORT_REPO_SUBDIR}')


## Run the model comparison

This is the main workflow cell. It reads the saved notebook-8 outputs, builds the comparison tables and plots, writes the HTML report, and optionally packages a clean public-export folder.

**Expected output**
- a list of saved comparison tables and plots
- a link to the full HTML report
- public-export dependency and sensitive-string scan results when export is enabled

**How to interpret the result**
- if the workflow fails immediately, review the run-spec paths from the previous cell
- dependency issues or sensitive-string matches should be resolved before any manual copy into the GitHub repository


In [ ]:
comparison_outputs = build_similarity_model_comparison(
    run_specs=RUN_SPECS,
    label_table_path=LABEL_TABLE_PATH,
    output_dir=OUTPUT_DIR,
    cloud_thresholds=CLOUD_THRESHOLDS,
    top_k_neighbors=TOP_K_NEIGHBORS,
    html_cloud_threshold=HTML_CLOUD_THRESHOLD,
    html_top_n_neighbors=HTML_TOP_N_NEIGHBORS,
    report_title=HTML_REPORT_TITLE,
    embed_html_images=EMBED_HTML_IMAGES,
)

comparison_tables = comparison_outputs['tables']

print('Saved comparison tables:')
for name, path in comparison_outputs['table_paths'].items():
    print(f'- {name}: {path}')

print('Saved comparison plots:')
for name, path in comparison_outputs['plot_paths'].items():
    print(f'- {name}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
print(f'HTML report: {comparison_outputs["plot_paths"]["html_report_path"]}')

display(HTML(
    f'<p><a href="{comparison_outputs["plot_paths"]["html_report_path"]}" target="_blank">Open full HTML comparison report</a></p>'
))

public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_similarity_model_comparison_html(
        comparison_outputs=comparison_outputs,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
    )

    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo report path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious personal paths or email strings were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before any GitHub copy step.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious personal or local-environment strings. Review the scan table before sharing the files.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so the notebook skipped the clean public-export step.')


## Review the whole-space distribution summary

This cell presents the broadest comparison view by summarizing the full all-vs-all similarity distribution for each model state.

**Expected output**
- a summary table for the all-vs-all distribution in each model
- the saved all-vs-all comparison plot

**How to interpret the result**
- differences here suggest that the global embedding geometry changed across model states, not just a few local neighborhoods


In [ ]:
all_vs_all_df = comparison_tables['all_vs_all_model_comparison']
summary_cols = ['model_id', 'model_label', 'num_glycans', 'count', 'mean', 'median', 'std_dev', 'q05', 'q25', 'q75', 'q95']
display(all_vs_all_df[summary_cols])
display(Image(filename=comparison_outputs['plot_paths']['all_vs_all_plot']))


## Review the query-specific summaries

This cell focuses on how the same query glycans behave across model states by comparing the saved specific-vs-all summaries.

**Expected output**
- a sorted query-summary table across all compared models
- the saved query-comparison plot

**How to interpret the result**
- large shifts in these summaries indicate that the neighborhood around one or more query glycans changed meaningfully across models


In [ ]:
specific_df = comparison_tables['specific_vs_all_model_comparison']
query_cols = ['model_id', 'model_label', 'query_accession', 'mean', 'median', 'std_dev', 'q05', 'q25', 'q75', 'q95', 'max']
display(specific_df[query_cols].sort_values(['query_accession', 'model_id']))
display(Image(filename=comparison_outputs['plot_paths']['specific_vs_all_plot']))


## Compare matched top-neighbor overlap

This cell compares the top-`k` neighborhoods returned for the same query glycans across model states.

**Expected output**
- a pairwise overlap table for every query glycan
- a three-way overlap summary table across the compared models

**How to interpret the result**
- low overlap means the neighborhood composition changed in a concrete, accession-level way


In [ ]:
top_neighbor_overlap_df = comparison_tables['top_neighbor_overlap_model_comparison']
top_neighbor_three_way_df = comparison_tables['top_neighbor_three_way_overlap_summary']

display(top_neighbor_overlap_df.sort_values(['query_accession', 'model_a', 'model_b']))
display(top_neighbor_three_way_df.sort_values(['query_accession']))


## Summarize top-neighbor stability across model pairs

This cell averages the pairwise top-neighbor overlap metrics across query glycans so the comparison can be scanned quickly at the model-pair level.

**Expected output**
- one summary row per model pair containing mean overlap counts and mean Jaccard overlap


In [ ]:
top_overlap_summary_df = (
    top_neighbor_overlap_df
    .groupby(['model_a_label', 'model_b_label'], as_index=False)[['overlap_count', 'jaccard_overlap']]
    .mean()
    .sort_values(['model_a_label', 'model_b_label'])
)
display(top_overlap_summary_df)


## Review neighborhood label overlap

If a label table is available, this cell summarizes how often the matched top-`k` neighborhoods share labels with the query glycans.

**Expected output**
- a detailed per-query label-overlap table when `LABEL_TABLE_PATH` exists
- an averaged label-overlap summary by model state

**How to interpret the result**
- these summaries are useful when the question is semantic rather than purely geometric: did nearby glycans remain label-consistent across model states?


In [ ]:
top_label_df = comparison_tables.get('top_neighbor_label_overlap_model_comparison')

if top_label_df is None:
    print('No label-overlap table was created because LABEL_TABLE_PATH was not set.')
else:
    review_cols = [
        'model_id',
        'model_label',
        'query_accession',
        'top_k',
        'query_labels_json',
        'neighborhood_size',
        'labeled_neighbors',
        'neighbors_without_labels',
        'exact_label_set_matches',
        'any_label_overlap',
        'no_label_overlap',
        'exact_label_set_match_rate',
        'any_label_overlap_rate',
    ]
    display(top_label_df[review_cols].sort_values(['query_accession', 'model_id']))

    top_label_summary_df = (
        top_label_df
        .groupby(['model_order', 'model_label'], as_index=False)[
            ['exact_label_set_match_rate', 'any_label_overlap_rate', 'neighborhood_size']
        ]
        .mean()
        .sort_values('model_order')
    )
    display(top_label_summary_df)
